In [43]:
import pandas as pd
import geopandas as gpd

In [44]:
world_countries = gpd.read_file("https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip")

In [45]:
world_countries.columns

Index(['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3',
       'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN',
       ...
       'FCLASS_TR', 'FCLASS_ID', 'FCLASS_PL', 'FCLASS_GR', 'FCLASS_IT',
       'FCLASS_NL', 'FCLASS_SE', 'FCLASS_BD', 'FCLASS_UA', 'geometry'],
      dtype='object', length=169)

In [46]:
print(world_countries['ADMIN'].nunique())

242


In [47]:
world_countries.columns

Index(['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3',
       'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN',
       ...
       'FCLASS_TR', 'FCLASS_ID', 'FCLASS_PL', 'FCLASS_GR', 'FCLASS_IT',
       'FCLASS_NL', 'FCLASS_SE', 'FCLASS_BD', 'FCLASS_UA', 'geometry'],
      dtype='object', length=169)

In [48]:
df = pd.read_parquet("../data/processed/immigration_data/immigration_stats_bc_census_subdivisions.parquet")

In [49]:
df["Place of birth (290)"].unique()

array(['Total – Place of birth', 'Inside Canada',
       'Newfoundland and Labrador', 'Nova Scotia', 'New Brunswick',
       'Quebec', 'Ontario', 'Manitoba', 'Saskatchewan', 'Alberta',
       'British Columbia', 'Northwest Territories', 'Outside Canada',
       'Americas', 'North America', 'United States of America',
       'Central America', 'Mexico', 'Europe', 'Western Europe',
       'Northern Europe', 'Ireland', 'United Kingdom', 'Southern Europe',
       'Asia', 'Eastern Asia', 'Southeast Asia', 'Philippines', 'Oceania',
       'New Zealand', 'Yukon', 'Caribbean and Bermuda', 'South America',
       'Germany', 'Netherlands', 'Eastern Europe', 'Poland', 'Italy',
       'Africa', 'Southern Africa', 'South Africa, Republic of',
       'Australia', 'Fiji', 'Prince Edward Island', 'Argentina', 'Brazil',
       'France', 'Czechia', 'Russian Federation', 'Sweden', 'Hong Kong',
       'Japan', 'Southern Asia', 'India', 'Nepal', 'El Salvador',
       'Jamaica', 'Ecuador', 'Peru', 'Austria'

In [50]:
import pandas as pd
import difflib

manual_mapping = {
    'United Republic of Tanzania': 'Tanzania',
    'South Korea': 'Korea, South',
    'North Korea': 'Korea, North',
    'Republic of the Congo': 'Congo, Republic of the',
    'Sint Maarten': 'Sint Maarten (Dutch part)',
    'Russia': 'Russian Federation',
    'Ivory Coast': "Côte d'Ivoire",
    'Republic of Serbia': 'Serbia',
    'East Timor': 'Timor-Leste',
    'Brunei': 'Brunei Darussalam',
    'eSwatini': 'Eswatini',
    'Hong Kong S.A.R.': 'Hong Kong',
    'Vatican': 'Holy See (Vatican City State)',
    'The Bahamas': 'Bahamas',
    'Pitcairn Islands': 'Pitcairn',
    'French Southern and Antarctic Lands': 'French Southern Territories',
    'United States Virgin Islands': 'Virgin Islands, United States',
    'British Virgin Islands': 'Virgin Islands, British',
    'Saint Helena': 'Saint Helena, Ascension and Tristan da Cunha',
    'Aland': 'Åland Islands',
    'Federated States of Micronesia': 'Micronesia, Federated States of',
    'South Georgia and the Islands': 'South Georgia and the South Sandwich Islands',
    'Macao S.A.R': 'Macao',
    'Saint Martin': 'Saint Martin (French part)'
    # 'Canada': 'Inside Canada'
}

# Get a list of names from the stats dataset to compare against
stats_names = df["Place of birth (290)"].unique()

# Function to apply fuzzy matching for names not exactly matching
def fuzzy_map(val, choices, cutoff=0.8):
    # Get the closest match from stats_names
    match = difflib.get_close_matches(val, choices, n=1, cutoff=cutoff)
    return match[0] if match else val

# Apply fuzzy matching for any values that didn't match exactly
world_countries['ADMIN'] = world_countries['ADMIN'].apply(lambda x: fuzzy_map(x, stats_names))

In [51]:
world_countries['ADMIN'] = world_countries['ADMIN'].replace(manual_mapping)

In [52]:
merged_df = pd.merge(
    world_countries,
    df,
    left_on='ADMIN',
    right_on='Place of birth (290)',
    how='left'
)

In [53]:
print(merged_df[['ADMIN', 'Place of birth (290)']].head())
print("Missing matches:", merged_df['Place of birth (290)'].isna().sum())

      ADMIN Place of birth (290)
0  Zimbabwe             Zimbabwe
1  Zimbabwe             Zimbabwe
2  Zimbabwe             Zimbabwe
3  Zimbabwe             Zimbabwe
4  Zimbabwe             Zimbabwe
Missing matches: 53


In [54]:
# After merging the dataframes:
missing_matches = merged_df[merged_df['Place of birth (290)'].isna()]['ADMIN'].unique()
print("Missing matches:", missing_matches)

Missing matches: ['Holy See (Vatican City State)' 'Vanuatu'
 'Micronesia, Federated States of' 'Marshall Islands'
 'Virgin Islands, United States' 'American Samoa'
 'South Georgia and the South Sandwich Islands'
 'British Indian Ocean Territory'
 'Saint Helena, Ascension and Tristan da Cunha' 'Pitcairn' 'Anguilla'
 'Falkland Islands' 'Turks and Caicos Islands' 'Solomon Islands'
 'São Tomé and Principe' 'San Marino' 'Palau' 'Niue' 'Cook Islands'
 'Nauru' 'Western Sahara' 'Mauritania' 'Liechtenstein' 'Kiribati'
 'Palestine' 'Guinea-Bissau' 'Gabon' 'Wallis and Futuna'
 'Saint Martin (French part)' 'Saint Barthelemy' 'New Caledonia'
 'French Southern Territories' 'Åland Islands' 'Equatorial Guinea'
 'Djibouti' 'Greenland' 'Faroe Islands' 'Northern Cyprus' 'Comoros'
 'Central African Republic' 'Cabo Verde' 'Canada' 'Burkina Faso' 'Benin'
 'Indian Ocean Territories' 'Heard Island and McDonald Islands'
 'Norfolk Island' 'Ashmore and Cartier Islands' 'Andorra'
 'Siachen Glacier' 'Antarctica' '

In [55]:
world_countries[['ADMIN', 'geometry']].to_file('../data/processed/geojson/world_countries_clean.geojson', driver='GeoJSON')

In [56]:
world_countries[world_countries['ADMIN']=='Hong Kong']

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
197,Admin-0 country,3,4,China,CH1,1,2,Country,1,Hong Kong,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((114.01543 22.51191, 114.01826 ..."
